# BA283 Neural Redesign — Full-Set Layered Ablation (Colab T4)

Phase-1 of the neural/data redesign. The fidelity-ablation matrix + C7 showed the
SPICE target is reproducible at **-42 dB** (not target-bound), so the **0.097**
`val_esr` wall is a model/training problem. This notebook trains a **cumulative**
stack of the four redesign levers on the **full** gain set (`ba283_spice_gain`,
not the tiny subsets) so the overfit confound that corrupted B0/C1/C4/C5 is gone,
and scores each step against the wall.

| cell | adds | flags |
|---|---|---|
| **A0** | baseline (reproduce the wall) | hidden 32, linear head/FiLM |
| **A1** | + multi-res STFT loss | `--stft-weight 0.5` |
| **A2** | + nonlinear output head | `--head mlp` |
| **A3** | + gain-MLP FiLM | `--film mlp` |
| **A4** | + capacity & reg | `--hidden 64 --layers 2 --dropout 0.1 --weight-decay 1e-5` |
| **A5** | + kind-balanced sampling | `--balance-by-kind` |

Each step is cumulative, so its row is the **marginal** contribution of that lever.
Trains on the **9-pt** set (`ba283_ablation/full9pt`): 5-pt gain grid in `train.csv`,
9-pt in `val.csv`, so the 4 interleaved gains are **held out**. Selection metric is the
**held-out-gain ESR** (FiLM interpolation on the 4 unseen gains) — the honest
generalization number; pooled/equal-weight `val_esr` is reported too but is
memorization-friendly (train/val share stems).

> Runtime: 6 full-set training runs on one T4 is multi-hour. Each cell checkpoints
> independently and the train loop **skips** any config whose `best.pth` already
> exists, so a runtime kill resumes cleanly — just re-run the train cell.

## 1. Install rclone + upload `rclone.conf`

In [ ]:
import os, subprocess
from google.colab import files

# install rclone (no-op if present)
if subprocess.run(['which','rclone'],capture_output=True).returncode:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash', shell=True, check=True)

# upload rclone.conf (token written to disk, not echoed)
cfg = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(cfg), exist_ok=True)
up = files.upload()  # pick your rclone.conf
assert up, 'no file uploaded'
open(cfg,'wb').write(next(iter(up.values())))
REMOTE = subprocess.run(['rclone','listremotes'],capture_output=True,text=True).stdout.split()[0]
print('remote:', REMOTE)

## 2. Clone the repo + deps + GPU check

In [ ]:
import getpass, subprocess
TOKEN = getpass.getpass('GitHub token: ')
subprocess.run(['git','clone','-q','https://x-access-token:%s@github.com/gustavokch/ba283-vst3.git'%TOKEN,'/content/ba283'])
%cd /content/ba283
!git pull -q
!pip -q install soundfile pandas tensorboard scipy
import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime type to T4 GPU'
print(torch.cuda.get_device_name(0))

## 3. Config — controls + the cumulative lever stack

In [ ]:
SEED, EPOCHS, PATIENCE = 745603, 60, 15
REFERENCE_WALL = 0.097          # the FiLM+LSTM full-set plateau we are trying to break
TARGET = 0.01                   # redesign goal (~ -20 dB), decision-rule reference
FULL = REMOTE + 'ba283_ablation/full9pt'       # 9-pt set: dry/ wet/ + train.csv val.csv manifest.csv
RESULTS = REMOTE + 'ba283_ablation/redesign'   # where the verdict is saved

# full9pt CONTRACT (differs from the old 6-pt ba283_spice_gain):
#   train.csv -> 5-pt gain grid {0, .25, .5, .75, 1.0}, 77 stems
#   val.csv   -> 9-pt gain grid {0, .125, ... 1.0}, SAME 77 stems
# The 4 interleaved gains {.125,.375,.625,.875} are HELD OUT of training, so
# run_cell's held_out_gain_esr scores FiLM gain-INTERPOLATION -- the honest
# generalization number here (stems overlap train/val, so pooled val_esr is
# memorization-friendly; select on held_out_gain_esr, report both).
GAINS = {0.0, 0.25, 0.5, 0.75, 1.0}            # the TRAINED gains (held-out = the 4 between)

# Stability knobs, layered with the fidelity levers (see plan move 3):
#  - STFT gradients are spikier than time-domain ESR -> tighten grad clip once
#    the spectral term is on (A1+).
#  - the big graph (hidden64/layers2 + STFT) is hot at the 5e-3 LSTM default ->
#    drop LR to 1e-3 for A4/A5 so BPTT does not burst.
_STAB_STFT = ['--clip-norm', '1.0']                  # A1..A3: STFT on, hidden32/1-layer
_STAB_BIG  = ['--clip-norm', '1.0', '--lr', '1e-3']  # A4..A5: + capacity

# Cumulative stack: each config = the previous one + one lever. 'flags' are extra
# train.py args; 'eval' are the BA283LSTM knobs run_cell must rebuild to load the
# checkpoint (must match the training architecture).
_REG = ['--hidden','64','--layers','2','--dropout','0.1','--weight-decay','1e-5']
_STK = ['--stft-weight','0.5','--head','mlp','--film','mlp']   # loss+head+gainMLP
CONFIGS = [
    dict(key='A0', label='baseline',  flags=['--hidden','32'],
         eval=dict(hidden=32)),
    dict(key='A1', label='+STFT',     flags=['--hidden','32','--stft-weight','0.5',*_STAB_STFT],
         eval=dict(hidden=32)),
    dict(key='A2', label='+head',     flags=['--hidden','32','--stft-weight','0.5','--head','mlp',*_STAB_STFT],
         eval=dict(hidden=32, head='mlp')),
    dict(key='A3', label='+gainMLP',  flags=['--hidden','32',*_STK,*_STAB_STFT],
         eval=dict(hidden=32, head='mlp', film_kind='mlp')),
    dict(key='A4', label='+capacity', flags=[*_REG,*_STK,*_STAB_BIG],
         eval=dict(hidden=64, head='mlp', film_kind='mlp', num_layers=2)),
    dict(key='A5', label='+balance',  flags=[*_REG,*_STK,*_STAB_BIG,'--balance-by-kind'],
         eval=dict(hidden=64, head='mlp', film_kind='mlp', num_layers=2)),
]
print('\n'.join(f"{c['key']:3} {c['label']:10} {' '.join(c['flags'])}" for c in CONFIGS))

## 4. Stage the full gain set (once) + remap manifests

In [ ]:
from train.stage import stage_rclone, remap_manifests, assert_matched_sr

DEST = '/content/full'
stage_rclone(FULL, DEST)                                   # rclone copy; asserts manifest.csv
MAN = remap_manifests(DEST, names=('train','val'), out_dir=DEST)   # -> {'train':csv,'val':csv}

# GUARD: dry SR == wet SR == 44100. The 96k-dry/44.1k-wet mismatch silently
# collapsed FULL9/C5 to zero-output (ESR 1.0) and only surfaced after a multi-hour
# run; fail loud here for a few WAV-header reads instead. (fix: data_gen PR #26)
sr = assert_matched_sr(MAN['train'], expected_sr=44100)
assert_matched_sr(MAN['val'], expected_sr=44100)
print(f"sample-rate guard OK: dry==wet=={sr} Hz")

import pandas as pd
for k,v in MAN.items():
    df = pd.read_csv(v)
    print(f"{k}: {len(df)} rows, {df.dry_path.nunique()} unique dry, gains {sorted(set(df.gain_value))}")
print(MAN)

## 5. Train the cumulative stack

Trains A0→A5 sequentially. A config whose `best.pth` already exists is skipped, so a
runtime kill just means re-running this cell.

In [ ]:
import os, subprocess

for cfg in CONFIGS:
    ck = f"ckpt_{cfg['key']}"
    if os.path.exists(f'{ck}/best.pth'):
        print('skip (done):', cfg['key']); continue
    cmd = ['python','-m','train.train',
           '--train-manifest', MAN['train'], '--val-manifest', MAN['val'],
           '--epochs', str(EPOCHS), '--patience', str(PATIENCE),
           '--conditioning','film','--device','cuda',
           '--checkpoint-dir', ck, '--log-dir', f"logs_{cfg['key']}",
           '--seed', str(SEED), *cfg['flags']]
    print('>>>', cfg['key'], cfg['label'], '|', ' '.join(cfg['flags']), flush=True)
    subprocess.run(cmd, check=True)

## 6. Score the stack + verdict table + save to Drive

`val_esr` is equal-weight (mean over the 6 gains). `ΔdB` is vs **A0**; a step
**explains** (✅) if it improves ≥3 dB or closes ≥50% of the A0→TARGET gap. The
`REFERENCE_WALL` line shows where A0 lands vs the historical 0.097 plateau.

In [ ]:
import json, os, subprocess
import numpy as np
from eval.ablation.run_checkpoint_eval import run_cell
from eval.ablation.report import render_table

results = []
for cfg in CONFIGS:
    r = run_cell(f"ckpt_{cfg['key']}/best.pth", MAN['val'], cell=cfg['key'],
                 conditioning='film', trained_gains=GAINS, **cfg['eval'])
    r['notes'] = cfg['label']
    results.append(r)
    hog = r.get('held_out_gain_esr')
    hog_s = ('hog=%.4f (%.1f dB)' % (hog, 10*np.log10(max(hog, 1e-12)))) if hog else 'hog=None'
    print('%-3s %-10s val_esr=%.4f (%.1f dB)  %s' %
          (cfg['key'], cfg['label'], r['val_esr'],
           10*np.log10(max(r['val_esr'], 1e-12)), hog_s))

base = next(r['val_esr'] for r in results if r['cell'] == 'A0')
for r in results:
    r['baseline_esr'] = base
table = render_table(results, target=TARGET)
print('\nSelect on HELD-OUT-GAIN esr (gain interpolation); val_esr is stem-overlap optimistic.')
print('A0=%.4f (%.1f dB)  vs REFERENCE_WALL=%.3f  TARGET=%.3f\n' %
      (base, 10*np.log10(max(base,1e-12)), REFERENCE_WALL, TARGET))
print(table)

os.makedirs('/content/redesign_save', exist_ok=True)
json.dump(results, open('/content/redesign_save/redesign_results.json','w'), indent=2)
open('/content/redesign_save/redesign_table.md','w').write(table)
subprocess.run(['rclone','copy','/content/redesign_save', RESULTS, '-P'], check=True)
print('saved ->', RESULTS)

Next: the winning levers (those clearing the gate) graduate to **Phase 2** — export
schema bump + the `plugin/src/model.rs` port + round-trip re-verify — so the win
ships in the realtime plugin. Re-run the winner on 2–3 seeds first to separate signal
from seed noise.